# Indexing Documents with Images [Step 2 - PDF Image Extraction and ChromaDB Storage]

> **MLCourse - Agentic AI - Multi-Modal RAG**

## What you will learn

1. How to extract images from a PDF using PyMuPDF.
2. How to embed both text chunks and extracted images using CLIP.
3. How to store multi-modal embeddings in ChromaDB with metadata.
4. The index structure that enables cross-modal retrieval.

Real documents contain text AND images -- diagrams, charts, figures.
A multi-modal RAG system must index both. This notebook extracts images
from "Attention Is All You Need" (the transformer paper), chunks the
text, embeds everything with CLIP, and stores it all in ChromaDB.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # silently skip the magic outside IPython

import matplotlib.pyplot as plt      # noqa: E402

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


In [2]:
# ## Part 1: Install Dependencies
#
# We need PyMuPDF (for PDF parsing), open_clip (for CLIP embeddings),
# ChromaDB (for vector storage), and Pillow (for image processing).

import subprocess, sys

def install_if_missing(package: str, import_name: str = None):
    """Install a package if it is not already importable."""
    name = import_name or package
    try:
        __import__(name)
        print(f"[OK] {package}")
    except ImportError:
        print(f"[INSTALL] {package} -- installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("pymupdf", "fitz")
install_if_missing("open-clip-torch", "open_clip")
install_if_missing("chromadb")
install_if_missing("Pillow")

[OK] pymupdf


[OK] open-clip-torch


[OK] chromadb
[INSTALL] Pillow -- installing...


In [3]:
# ## Part 2: Extract Images from the PDF
#
# We use PyMuPDF (imported as fitz) to iterate over every page of the
# "Attention Is All You Need" paper and extract embedded images.

import fitz                          # PyMuPDF -- fast PDF parser
from PIL import Image
import io

PDF_PATH = DATA / "attention_is_all_you_need.pdf"
EXTRACT_DIR = DATA / "multimodal_images"
EXTRACT_DIR.mkdir(exist_ok=True)

doc = fitz.open(str(PDF_PATH))
print(f"[PDF] Opened: {PDF_PATH.name}")
print(f"[PDF] Pages: {len(doc)}")

extracted_images = []
for page_num in range(len(doc)):
    page = doc[page_num]
    image_list = page.get_images(full=True)
    for img_idx, img_info in enumerate(image_list):
        xref = img_info[0]
        base_image = doc.extract_image(xref)
        img_bytes = base_image["image"]
        ext = base_image["ext"]

        # Save image to disk.
        img_name = f"page{page_num + 1}_img{img_idx + 1}.{ext}"
        img_path = EXTRACT_DIR / img_name
        with open(img_path, "wb") as f:
            f.write(img_bytes)

        # Open with Pillow to check dimensions.
        img = Image.open(io.BytesIO(img_bytes))
        w, h = img.size
        # Skip tiny images (likely bullets, lines, or artifacts).
        if w < 50 or h < 50:
            img_path.unlink()        # remove the saved file
            continue

        extracted_images.append({
            "path": str(img_path),
            "name": img_name,
            "page": page_num + 1,
            "width": w,
            "height": h,
        })

doc.close()
print(f"\n[EXTRACT] Found {len(extracted_images)} images (>= 50x50 px)")
for info in extracted_images:
    print(f"  {info['name']:<30} page {info['page']:<3}  {info['width']}x{info['height']}")

[PDF] Opened: attention_is_all_you_need.pdf
[PDF] Pages: 15

[EXTRACT] Found 3 images (>= 50x50 px)
  page3_img1.png                 page 3    1520x2239
  page4_img1.png                 page 4    445x884
  page4_img2.png                 page 4    835x1282


In [4]:
# ## Part 3: Extract Text from the PDF
#
# We extract text page-by-page and split into chunks using a simple
# paragraph-based strategy. Each chunk gets page metadata.

doc = fitz.open(str(PDF_PATH))

text_chunks = []
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text")      # plain text extraction

    # Split by double newlines (paragraph boundaries).
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    for para_idx, para in enumerate(paragraphs):
        # Skip very short paragraphs (likely parsing artifacts).
        if len(para) < 20:
            continue
        text_chunks.append({
            "text": para,
            "page": page_num + 1,
            "chunk_id": f"page{page_num + 1}_para{para_idx}",
            "source": PDF_PATH.name,
        })

doc.close()
print(f"[TEXT] Extracted {len(text_chunks)} text chunks from {PDF_PATH.name}")
print(f"[TEXT] Sample chunk (page {text_chunks[0]['page']}):")
print(f"  {text_chunks[0]['text'][:120]}...")

[TEXT] Extracted 15 text chunks from attention_is_all_you_need.pdf
[TEXT] Sample chunk (page 1):
  Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this pap...


In [5]:
# ## Part 4: Load CLIP Model for Embedding
#
# We reuse the same ViT-B-32 CLIP model from notebook 01 to embed
# both text chunks and extracted images into the shared space.

import torch
import open_clip

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device).eval()

print(f"[CLIP] Model loaded on {device}, dim={model.visual.output_dim}")

[CLIP] Model loaded on cpu, dim=512


In [6]:
# ## Part 5: Embed Text Chunks with CLIP
#
# We embed all text chunks in batches for efficiency. Each embedding is
# L2-normalized so cosine similarity equals dot product.

def embed_texts(texts: list[str], batch_size: int = 32) -> torch.Tensor:
    """Embed a list of texts with CLIP, returning L2-normalized vectors."""
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        tokens = tokenizer(batch).to(device)
        with torch.no_grad():
            vecs = model.encode_text(tokens)
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        all_vecs.append(vecs.cpu())
    return torch.cat(all_vecs, dim=0)

all_text = [c["text"] for c in text_chunks]
text_embeddings = embed_texts(all_text)

print(f"[EMBED] Embedded {len(all_text)} text chunks")
print(f"[EMBED] Shape: {text_embeddings.shape}")

[EMBED] Embedded 15 text chunks
[EMBED] Shape: torch.Size([15, 512])


In [7]:
# ## Part 6: Embed Extracted Images with CLIP
#
# Each extracted image is preprocessed (resized, normalized) and passed
# through CLIP's visual encoder. Same normalization as text.

def embed_images(img_paths: list[str], batch_size: int = 8) -> torch.Tensor:
    """Embed a list of image paths with CLIP, returning L2-normalized vectors."""
    tensors = []
    for path in img_paths:
        img = Image.open(path).convert("RGB")
        tensors.append(preprocess(img).unsqueeze(0))

    all_vecs = []
    for i in range(0, len(tensors), batch_size):
        batch = torch.cat(tensors[i:i + batch_size], dim=0).to(device)
        with torch.no_grad():
            vecs = model.encode_image(batch)
            vecs = vecs / vecs.norm(dim=-1, keepdim=True)
        all_vecs.append(vecs.cpu())
    return torch.cat(all_vecs, dim=0)

img_paths = [info["path"] for info in extracted_images]
if img_paths:
    image_embeddings = embed_images(img_paths)
    print(f"[EMBED] Embedded {len(img_paths)} images")
    print(f"[EMBED] Shape: {image_embeddings.shape}")
else:
    image_embeddings = torch.zeros(0, model.visual.output_dim)
    print("[EMBED] No images to embed")

[EMBED] Embedded 3 images
[EMBED] Shape: torch.Size([3, 512])


In [8]:
# ## Part 7: Store Everything in ChromaDB
#
# ChromaDB stores vectors with metadata. We create TWO collections:
#   1. "multimodal_text" -- text chunk embeddings + metadata
#   2. "multimodal_images" -- image embeddings + metadata
#
# Both use the same CLIP embedding space, enabling cross-modal queries.

import chromadb
import numpy as np

CHROMA_DIR = TRACK / "data" / "chroma_multimodal"
CHROMA_DIR.mkdir(exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Delete old collections if they exist (clean rebuild).
try:
    client.delete_collection("multimodal_text")
    client.delete_collection("multimodal_images")
except Exception:
    pass

text_collection = client.create_collection(
    name="multimodal_text",
    metadata={"hnsw:space": "cosine"}    # cosine distance metric
)
image_collection = client.create_collection(
    name="multimodal_images",
    metadata={"hnsw:space": "cosine"}
)

# Add text embeddings.
text_collection.add(
    ids=[c["chunk_id"] for c in text_chunks],
    embeddings=text_embeddings.numpy().tolist(),
    documents=all_text,
    metadatas=[{
        "page": c["page"],
        "source": c["source"],
        "type": "text",
    } for c in text_chunks]
)

print(f"[CHROMA] Added {text_collection.count()} text chunks to 'multimodal_text'")

# Add image embeddings (if any).
if img_paths:
    image_collection.add(
        ids=[info["name"] for info in extracted_images],
        embeddings=image_embeddings.numpy().tolist(),
        documents=[info["name"] for info in extracted_images],
        metadatas=[{
            "page": info["page"],
            "path": info["path"],
            "width": info["width"],
            "height": info["height"],
            "type": "image",
        } for info in extracted_images]
    )
    print(f"[CHROMA] Added {image_collection.count()} images to 'multimodal_images'")
else:
    print("[CHROMA] No images to add")

[CHROMA] Added 15 text chunks to 'multimodal_text'
[CHROMA] Added 3 images to 'multimodal_images'


In [9]:
# ## Part 8: Verify the Index
#
# Let us confirm the collections contain the right data and run a
# quick sanity-check query.

# Verify counts.
print(f"[VERIFY] Text collection:  {text_collection.count()} documents")
print(f"[VERIFY] Image collection: {image_collection.count()} documents")

# Quick text query to confirm retrieval works.
query = "attention mechanism in transformers"
query_tokens = tokenizer([query]).to(device)
with torch.no_grad():
    query_vec = model.encode_text(query_tokens)
    query_vec = query_vec / query_vec.norm(dim=-1, keepdim=True)

results = text_collection.query(
    query_embeddings=query_vec.numpy().tolist(),
    n_results=3,
)

print(f"\n[QUERY] \"{query}\"")
print("-" * 60)
for rank, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
), 1):
    print(f"  #{rank} (page {meta['page']}, dist={dist:.4f})")
    print(f"      {doc[:100]}...")

# Quick image query to confirm cross-modal works.
if img_paths:
    query2 = "self-attention visualization"
    query2_tokens = tokenizer([query2]).to(device)
    with torch.no_grad():
        q2_vec = model.encode_text(query2_tokens)
        q2_vec = q2_vec / q2_vec.norm(dim=-1, keepdim=True)

    img_results = image_collection.query(
        query_embeddings=q2_vec.numpy().tolist(),
        n_results=2,
    )
    print(f"\n[QUERY] \"{query2}\" -> image results")
    for rank, (doc, meta) in enumerate(zip(
        img_results["documents"][0],
        img_results["metadatas"][0],
    ), 1):
        print(f"  #{rank} {doc} (page {meta['page']})")

[VERIFY] Text collection:  15 documents
[VERIFY] Image collection: 3 documents

[QUERY] "attention mechanism in transformers"
------------------------------------------------------------
  #1 (page 2, dist=0.4119)
      1
Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural...
  #2 (page 7, dist=0.4380)
      length n is smaller than the representation dimensionality d, which is most often the case with
sent...
  #3 (page 1, dist=0.4635)
      Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and...

[QUERY] "self-attention visualization" -> image results
  #1 page4_img2.png (page 4)
  #2 page4_img1.png (page 4)


In [10]:
# ## Part 9: Summary of the Index Structure
#
# The multi-modal index looks like this:
#
#     CLIP text encoder -> text_embeddings -> ChromaDB "multimodal_text"
#     CLIP image encoder -> image_embeddings -> ChromaDB "multimodal_images"
#
# Both collections share the same embedding dimension (512 for ViT-B-32).
# A text query can search BOTH collections. An image query can search
# BOTH collections. This is the foundation of cross-modal retrieval.

print("\nIndex structure:")
print("=" * 60)
print(f"  Text collection:  {text_collection.count()} chunks")
print(f"  Image collection: {image_collection.count()} images")
print(f"  Embedding model:  CLIP ViT-B-32 (512 dims)")
print(f"  Distance metric:  Cosine")
print(f"  Storage:          {CHROMA_DIR}")


Index structure:
  Text collection:  15 chunks
  Image collection: 3 images
  Embedding model:  CLIP ViT-B-32 (512 dims)
  Distance metric:  Cosine
  Storage:          D:\projects\python\MLCourse\03_agentic_ai\data\chroma_multimodal


In [11]:
# ## Summary
#
# 1. PyMuPDF extracts images from PDF pages (filtering tiny artifacts).
# 2. Text is split into paragraph-level chunks with page metadata.
# 3. CLIP embeds both text and images into a shared 512-dim space.
# 4. ChromaDB stores both in separate collections with metadata.
# 5. A text query can retrieve from both text and image collections.
#
# Next: notebook 03 demonstrates cross-modal retrieval in detail.

print("\n[COMPLETE] Module 21 Notebook 2: Multimodal Indexing")
print("  - PDF images extracted:", len(extracted_images))
print("  - Text chunks created:", len(text_chunks))
print("  - ChromaDB index at:", CHROMA_DIR)


[COMPLETE] Module 21 Notebook 2: Multimodal Indexing
  - PDF images extracted: 3
  - Text chunks created: 15
  - ChromaDB index at: D:\projects\python\MLCourse\03_agentic_ai\data\chroma_multimodal
